In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import keras
import keras.backend as K
from keras import optimizers
from keras.layers import (Activation, Dense, Dropout)
from keras.layers.normalization import BatchNormalization
from keras.models import Sequential
import argparse
import pickle
import copy 
from scipy import spatial
import tensorflow as tf


Using TensorFlow backend.
/home/siddharth/Siddharth/Compiler/Adversaries-in-IR/pyenv/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:493: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/siddharth/Siddharth/Compiler/Adversaries-in-IR/pyenv/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:494: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/siddharth/Siddharth/Compiler/Adversaries-in-IR/pyenv/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:495: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.

ModuleNotFoundError: No module named 'tensorflow.keras'

In [10]:
from tensorflow.python.keras.losses import SparseCategoricalCrossentropy

ImportError: cannot import name 'SparseCategoricalCrossentropy'

In [2]:
X = pd.read_csv('./data/testing.csv', sep='\t',header=None)
Y = X.loc[:,0]
X = X.loc[:,1:]
X.columns = range(X.shape[1])

In [3]:
with open('./data/dictionary.pkl', 'rb') as f:
    num_classes = pickle.load(f)
    X_min = pickle.load(f)
    X_max = pickle.load(f)
    ipca=pickle.load(f)
    
x_train = (X - X_min) / (X_max - X_min)
x_train = np.array(x_train)
y_train = np.array(Y)
y_train = y_train - 1
y_train = keras.utils.to_categorical(y_train, num_classes)
x_train = ipca.transform(x_train)

In [4]:
model = keras.models.load_model("./data/last-model.h5")
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_1 (Dense)              (None, 650)               195650    
_________________________________________________________________
batch_normalization_1 (Batch (None, 650)               2600      
_________________________________________________________________
activation_1 (Activation)    (None, 650)               0         
_________________________________________________________________
dropout_1 (Dropout)          (None, 650)               0         
_________________________________________________________________
dense_2 (Dense)              (None, 600)               390600    
_________________________________________________________________
batch_normalization_2 (Batch (None, 600)               2400      
_________________________________________________________________
activation_2 (Activation)    (None, 600)               0         
__________

In [5]:
score = model.evaluate(x_train, y_train, verbose=0)
print('Test accuracy : {acc:.3f}%'.format(acc=score[1]*100))

Test accuracy : 89.998%


In [6]:
target=model.predict_classes(x_train, batch_size=32, verbose=0)
x_new = []
y_new = []
correct_out = Y -1
for i in range(target.shape[0]):
    if target[i] == correct_out[i]:
        x_new.append(x_train[i])
        y_new.append(target[i])

x_new = np.array(x_new)
y_new = np.array(y_new)
y_new_categorical = keras.utils.to_categorical(y_new, num_classes)
print(x_new.shape, y_new.shape, y_new_categorical.shape)
model.trainable = False

(7450, 300) (7450,) (7450, 104)


In [7]:
learning_rate = .00001
#value of gradient for the first x_test
x_test_1 = copy.deepcopy(x_new[1:2])
x_orignal = copy.deepcopy(x_new[1:2])

out = (y_new[2] + 10) % 104
out = keras.utils.to_categorical(out, num_classes)

step = 10
sccLoss = SparseCategoricalCrossentropy()

NameError: name 'SparseCategoricalCrossentropy' is not defined

In [ ]:
for step in range(0, steps):
    # record our gradients
    with tf.GradientTape() as tape:
        # explicitly indicate that our perturbation vector should
        # be tracked for gradient updates
        tape.watch(x_test_1)
        # add our perturbation vector to the base image and
        # preprocess the resulting image
        adversary = x_test_1
        # run this newly constructed image tensor through our
        # model and calculate the loss with respect to the
        # both the *original* class label and the *target*
        # class label
        predictions = model(adversary, training=False)
        originalLoss = -sccLoss(tf.convert_to_tensor(y_new[2]),
            predictions)
        targetLoss = sccLoss(tf.convert_to_tensor(out),
            predictions)
        totalLoss = originalLoss + targetLoss
        # check to see if we are logging the loss value, and if
        # so, display it to our terminal
        if step % 2 == 0:
            print("step: {}, loss: {}...".format(step,
                totalLoss.numpy()))
    # calculate the gradients of loss with respect to the
    # perturbation vector
    gradients = tape.gradient(totalLoss, x_test_1)
    # update the weights, clip the perturbation vector, and
    # update its value
    optimizer.apply_gradients([(gradients, x_test_1)])
#     delta.assign_add(clip_eps(delta, eps=EPS))
# return the perturbation vector
# return delta

In [ ]:
orignal_y = y_new[2]
predicted_y = model.predict_classes(x_test_1, batch_size=32, verbose=2)
print(orignal_y, predicted_y, np.argmax(out))
result = 1 - spatial.distance.cosine(x_test_1, x_orignal)
print(result)

In [10]:
orignal_y = y_new[2]
sess = K.get_session()
print(type(model.output))
for i in range(10):
#     out = out.astype('float32')
#     out = K.constant(out)
    loss = keras.losses.categorical_crossentropy(model.output,out)
    gradients = K.gradients(loss, model.input)
    evaluated_gradients_1 = sess.run(gradients[0], feed_dict={model.input: 
    x_test_1})    
    x_test_1 -= learning_rate*evaluated_gradients_1
    predicted_y = model.predict_classes(x_test_1, batch_size=32, verbose=2)
    print(orignal_y, predicted_y, np.argmax(out))
    result = 1 - spatial.distance.cosine(x_test_1, x_orignal)
    print(result)
    loss_print =  keras.backend.print_tensor(loss, message='loss')
#     loss = tf.Print(loss, [loss])
    print(loss_print)

<class 'tensorflow.python.framework.ops.Tensor'>


AttributeError: 'numpy.ndarray' object has no attribute 'get_shape'

In [ ]:
result = 1 - spatial.distance.cosine(x_test_1, x_orignal)
print(result)